<a href="https://colab.research.google.com/github/HariPrasad017/ml-leakage-pipeline-Hari_Prasad_S_S/blob/main/ml_leakage_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1 — What I’m Trying to Do

So here, my goal is to evaluate a classification model properly and make sure I’m not accidentally leaking information from test data into training.

I’ll first do it the wrong way (to understand leakage) and then fix it using a proper pipeline.

# Task 1 — Reproduce and Identify Leakage

In [1]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Create the dataset
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)

# WRONG: Scaling before split (this causes leakage)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Train the model
model = LogisticRegression()
model.fit(X_train, y_train)

# Predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Accuracy of Model
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

Train Accuracy: 0.87
Test Accuracy: 0.83


## What Went Wrong?

Honestly, this approach looks fine at first… but it’s actually wrong.

- I scaled the entire dataset before splitting.
That means information from the test set is already used while computing mean and standard deviation.

- In simple words:
The model has indirectly “seen” the test data.

- This leads to:

    - Over-optimistic performance
    - Unrealistic results

# Task 2 — Fix Using Pipeline + Cross Validation

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# Create pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

# Split first (correct way)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Cross-validation
scores = cross_val_score(pipeline, X_train, y_train, cv=5)

print("Cross-validation scores:", scores)
print("Mean Accuracy:", scores.mean())
print("Standard Deviation:", scores.std())

Cross-validation scores: [0.88125 0.89375 0.875   0.84375 0.85625]
Mean Accuracy: 0.8699999999999999
Standard Deviation: 0.01785357107135714


## Why This Is Correct

Now I’m doing things properly:

- Scaling happens inside the pipeline
- For each fold, scaling is done only on training data
- No leakage at all

Cross-validation also gives a better idea of model stability.





# Task 3 — Decision Tree Depth Experiment

In [4]:
from sklearn.tree import DecisionTreeClassifier

depths = [1, 5, 20]

results = []

for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)

    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)

    results.append((depth, train_acc, test_acc))

# Display results
for r in results:
    print(f"Depth: {r[0]}, Train Acc: {r[1]:.3f}, Test Acc: {r[2]:.3f}")

Depth: 1, Train Acc: 0.882, Test Acc: 0.840
Depth: 5, Train Acc: 0.954, Test Acc: 0.855
Depth: 20, Train Acc: 1.000, Test Acc: 0.840


| Depth | Train Accuracy | Test Accuracy |
| ----- | -------------- | ------------- |
| 1     | ~0.882          | ~0.840         |
| 5     | ~0.954          | ~0.855         |
| 20    | ~1.000          | ~0.840         |
